# Silver Layer - Cleaned and Typed Data
### Egyptian Investment Intelligence Platform

Goal: Clean, cast types, drop useless columns, and standardize all bronze tables. No business aggregations.

| Silver Table | Source | Key Transformations |
|---|---|---|
| silver.stocks_eod | bronze.stocks_eod | Snake_case, cast date, drop Dividends/Stock_Splits, dedup |
| silver.egx30_index | bronze.egx30_index | Snake_case, cast date, dedup |
| silver.fundamentals | bronze.fundamentals | Drop website/country/exchange/currency, snake_case |
| silver.live_quotes | bronze.live_quotes | Drop currency/exchange, rename, round floats |
| silver.gold_silver_prices | bronze.gold_silver_prices | Cast date, uppercase metal, round price, dedup |
| silver.currency_rates | bronze.currency_rates | Filter to 10 relevant currencies only |
| silver.spot_prices | bronze.spot_prices | Uppercase metal, round all price fields |
| silver.real_estate_propertyfinder | bronze.real_estate_enriched | Parse area/price to numeric, drop image/link/redundant cols |
| silver.real_estate_unified | silver.real_estate_propertyfinder | Final unified table for gold layer |

## Step 1 - Spark Session

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, regexp_replace,
    upper, round as spark_round, array_join, to_timestamp
)
from pyspark.sql.types import DoubleType, IntegerType, LongType, DateType

spark = SparkSession.builder \
    .appName('Silver_Layer') \
    .config('spark.jars.packages',
            'org.apache.iceberg:iceberg-spark-runtime-3.3_2.12:1.3.0,'
            'org.projectnessie.nessie-integrations:nessie-spark-extensions-3.3_2.12:0.67.0,'
            'software.amazon.awssdk:url-connection-client:2.20.18') \
    .config('spark.sql.extensions',
            'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,'
            'org.projectnessie.spark.extensions.NessieSparkSessionExtensions') \
    .config('spark.sql.catalog.nessie', 'org.apache.iceberg.spark.SparkCatalog') \
    .config('spark.sql.catalog.nessie.catalog-impl', 'org.apache.iceberg.nessie.NessieCatalog') \
    .config('spark.sql.catalog.nessie.uri', 'http://nessie:19120/api/v1') \
    .config('spark.sql.catalog.nessie.ref', 'main') \
    .config('spark.sql.catalog.nessie.warehouse', 's3://my-icebergdatalake/iceberg-warehouse/') \
    .config('spark.sql.catalog.nessie.io-impl', 'org.apache.iceberg.aws.s3.S3FileIO') \
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem') \
    .config('spark.hadoop.fs.s3a.aws.credentials.provider',
            'com.amazonaws.auth.EnvironmentVariableCredentialsProvider') \
    .config('spark.hadoop.fs.s3a.access.key', os.environ['AWS_ACCESS_KEY_ID']) \
    .config('spark.hadoop.fs.s3a.secret.key', os.environ['AWS_SECRET_ACCESS_KEY']) \
    .config('spark.hadoop.fs.s3a.endpoint.region', 'eu-north-1') \
    .config('spark.driver.host', 'localhost') \
    .config('spark.driver.bindAddress', '0.0.0.0') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print('Spark Session started successfully')

## Step 2 - Ensure Silver Namespace and Helper

In [ ]:
spark.sql('CREATE NAMESPACE IF NOT EXISTS nessie.silver')
print('Silver namespace ready')

def write_silver(df, table_name):
    df = df.withColumn('_silver_at', current_timestamp())
    df.writeTo(f'nessie.silver.{table_name}') \
        .tableProperty('location', f's3://my-icebergdatalake/silver/{table_name}') \
        .createOrReplace()
    count = df.count()
    print(f'silver.{table_name} - {count} rows written')
    return count

print('Helper defined')

## Table 1 - silver.stocks_eod
Drops: Dividends, Stock_Splits, _source_file. Renames to snake_case. Deduplicates on (symbol, date).

In [ ]:
df = spark.table('nessie.bronze.stocks_eod')

silver_stocks = df \
    .select(
        col('Symbol').alias('symbol'),
        col('Date').cast(DateType()).alias('date'),
        col('Open').alias('open'),
        col('High').alias('high'),
        col('Low').alias('low'),
        col('Close').alias('close'),
        col('Volume').cast(LongType()).alias('volume'),
        col('_ingested_at')
    ) \
    .filter(col('close').isNotNull()) \
    .filter(col('symbol').isNotNull()) \
    .dropDuplicates(['symbol', 'date'])

write_silver(silver_stocks, 'stocks_eod')
silver_stocks.printSchema()
silver_stocks.show(3)

## Table 2 - silver.egx30_index
Drops: _source_file. Renames to snake_case. Deduplicates on date.

In [ ]:
df = spark.table('nessie.bronze.egx30_index')

silver_egx30 = df \
    .select(
        col('Date').cast(DateType()).alias('date'),
        col('Open').alias('open'),
        col('High').alias('high'),
        col('Low').alias('low'),
        col('Close').alias('close'),
        col('Volume').cast(LongType()).alias('volume'),
        col('_ingested_at')
    ) \
    .filter(col('close').isNotNull()) \
    .dropDuplicates(['date'])

write_silver(silver_egx30, 'egx30_index')
silver_egx30.printSchema()
silver_egx30.show(3)

## Table 3 - silver.fundamentals
Drops: Website, Country (all Egypt), Exchange (all EGX), Currency (all EGP), _source_file.

In [ ]:
df = spark.table('nessie.bronze.fundamentals')

silver_fundamentals = df \
    .select(
        col('Symbol').alias('symbol'),
        col('Name').alias('company_name'),
        col('Sector').alias('sector'),
        col('Industry').alias('industry'),
        col('MarketCap').cast(DoubleType()).alias('market_cap_egp'),
        col('TrailingPE').alias('trailing_pe'),
        col('ForwardPE').alias('forward_pe'),
        col('PriceToBook').alias('price_to_book'),
        col('EPS').alias('eps'),
        col('DividendYield').alias('dividend_yield'),
        col('Beta').alias('beta'),
        col('FiftyTwoWeekHigh').alias('week52_high'),
        col('FiftyTwoWeekLow').alias('week52_low'),
        col('Employees').alias('employees'),
        col('_ingested_at')
    ) \
    .filter(col('symbol').isNotNull())

write_silver(silver_fundamentals, 'fundamentals')
silver_fundamentals.printSchema()
silver_fundamentals.show(3)

## Table 4 - silver.live_quotes
Drops: Currency (all EGP), Exchange (all EGX), _source_file. Rounds floats to 4 decimal places.

In [ ]:
df = spark.table('nessie.bronze.live_quotes')

silver_live = df \
    .select(
        col('Symbol').alias('symbol'),
        col('Timestamp').cast(DateType()).alias('date'),
        spark_round(col('previousClose'), 4).alias('prev_close'),
        spark_round(col('Open'), 4).alias('open'),
        spark_round(col('dayHigh'), 4).alias('day_high'),
        spark_round(col('dayLow'), 4).alias('day_low'),
        spark_round(col('Close'), 4).alias('close'),
        col('Volume').cast(LongType()).alias('volume'),
        spark_round(col('Change'), 4).alias('change'),
        spark_round(col('Change_pct'), 4).alias('change_pct'),
        col('MarketCap').cast(DoubleType()).alias('market_cap_egp'),
        col('_ingested_at')
    ) \
    .filter(col('symbol').isNotNull())

write_silver(silver_live, 'live_quotes')
silver_live.printSchema()
silver_live.show(3)

## Table 5 - silver.gold_silver_prices
Drops: authority (all lbma), _source_file. Uppercases metal. Deduplicates on (date, metal, session).

In [ ]:
df = spark.table('nessie.bronze.gold_silver_prices')

silver_metals = df \
    .select(
        col('time').cast(DateType()).alias('date'),
        upper(col('metal')).alias('metal'),
        col('session'),
        col('unit'),
        spark_round(col('price'), 6).alias('price_usd'),
        col('currency'),
        col('_ingested_at')
    ) \
    .filter(col('price').isNotNull()) \
    .filter(col('price') > 0) \
    .dropDuplicates(['date', 'metal', 'session'])

write_silver(silver_metals, 'gold_silver_prices')
silver_metals.printSchema()
silver_metals.show(3)

## Table 6 - silver.currency_rates
Filters to 10 investment-relevant currencies only: USD, EUR, GBP, JPY, CNY, SAR, AED, EGP, CHF, CAD.

In [ ]:
df = spark.table('nessie.bronze.currency_rates')

relevant_currencies = ['USD', 'EUR', 'GBP', 'JPY', 'CNY', 'SAR', 'AED', 'EGP', 'CHF', 'CAD']

silver_currency = df \
    .select(
        col('time').alias('snapshot_time'),
        upper(col('currency')).alias('currency'),
        spark_round(col('rate'), 8).alias('rate_vs_usd'),
        col('_ingested_at')
    ) \
    .filter(upper(col('currency')).isin(relevant_currencies)) \
    .filter(col('rate').isNotNull()) \
    .filter(col('rate') > 0)

write_silver(silver_currency, 'currency_rates')
silver_currency.printSchema()
silver_currency.show(10)

## Table 7 - silver.spot_prices
Uppercases metal. Rounds all price fields to 6 decimal places.

In [ ]:
df = spark.table('nessie.bronze.spot_prices')

silver_spot = df \
    .select(
        col('time').alias('snapshot_time'),
        upper(col('metal')).alias('metal'),
        col('currency'),
        col('unit'),
        spark_round(col('price'), 6).alias('price'),
        spark_round(col('bid'), 6).alias('bid'),
        spark_round(col('ask'), 6).alias('ask'),
        spark_round(col('high'), 6).alias('high'),
        spark_round(col('low'), 6).alias('low'),
        spark_round(col('change'), 4).alias('change'),
        spark_round(col('change_pct'), 4).alias('change_pct'),
        col('_ingested_at')
    ) \
    .filter(col('price').isNotNull())

write_silver(silver_spot, 'spot_prices')
silver_spot.printSchema()
silver_spot.show(3)

## Table 8 - silver.real_estate_propertyfinder
Drops: image, link, subtitle, full_title, property_size_full, bathrooms_full, bedrooms_full, property_type_full, _source_file.
Parses area from '165 sqm' to 165.0. Parses price from '5,900,000' to 5900000.0. Converts amenities array to string. Deduplicates on listing_id.

In [ ]:
df = spark.table('nessie.bronze.real_estate_enriched')

silver_pf = df \
    .select(
        col('listing_id'),
        col('property_type'),
        col('title'),
        col('location'),
        col('full_location'),
        col('bedrooms').cast(IntegerType()).alias('bedrooms'),
        col('bathrooms').cast(IntegerType()).alias('bathrooms'),
        regexp_replace(col('area'), r'[^0-9.]', '').cast(DoubleType()).alias('area_sqm'),
        regexp_replace(col('price_full'), r'[^0-9.]', '').cast(DoubleType()).alias('price_egp'),
        regexp_replace(col('price_per_sqm'), r'[^0-9.]', '').cast(DoubleType()).alias('price_per_sqm'),
        col('listing_level'),
        col('project_name'),
        col('project_status'),
        col('developer'),
        col('delivery_date'),
        col('agent_name'),
        col('broker_name'),
        col('regulatory_reference'),
        array_join(col('amenities'), ', ').alias('amenities'),
        to_timestamp(col('scraped_at')).alias('scraped_at'),
        lit('propertyfinder').alias('source'),
        col('_ingested_at')
    ) \
    .filter(col('price_egp').isNotNull()) \
    .filter(col('price_egp') > 0) \
    .filter(col('area_sqm').isNotNull()) \
    .filter(col('area_sqm') > 0) \
    .dropDuplicates(['listing_id'])

write_silver(silver_pf, 'real_estate_propertyfinder')
silver_pf.printSchema()
silver_pf.show(3)

## Table 9 - silver.real_estate_unified
Final unified real estate table for the gold layer. Source: PropertyFinder only (Bayut data was too sparse for analysis).

In [ ]:
pf = spark.table('nessie.silver.real_estate_propertyfinder')

silver_unified = pf \
    .select(
        col('listing_id'),
        col('property_type'),
        col('location'),
        col('full_location'),
        col('bedrooms'),
        col('bathrooms'),
        col('area_sqm'),
        col('price_egp'),
        col('price_per_sqm'),
        col('project_status'),
        col('developer'),
        col('delivery_date'),
        col('project_name'),
        col('amenities'),
        col('source'),
        col('_ingested_at')
    )

write_silver(silver_unified, 'real_estate_unified')
silver_unified.printSchema()
silver_unified.show(3)

## Final Verification - All Silver Tables

In [ ]:
print('=' * 60)
print('  SILVER LAYER - FINAL VERIFICATION')
print('=' * 60)

silver_tables = [
    'nessie.silver.stocks_eod',
    'nessie.silver.egx30_index',
    'nessie.silver.fundamentals',
    'nessie.silver.live_quotes',
    'nessie.silver.gold_silver_prices',
    'nessie.silver.currency_rates',
    'nessie.silver.spot_prices',
    'nessie.silver.real_estate_propertyfinder',
    'nessie.silver.real_estate_unified',
]

total = 0
for table in silver_tables:
    try:
        count = spark.table(table).count()
        total += count
        name = table.split('.')[-1]
        print(f'   {name:<42} {count:>7} rows')
    except Exception as e:
        name = table.split('.')[-1]
        print(f'   {name:<42}   ERROR: {e}')

print('=' * 60)
print(f'   TOTAL ROWS: {total}')
print('=' * 60)
print('Silver Layer complete.')
print('Stored at  : s3://my-icebergdatalake/silver/')
print('Tracked by : Nessie -> nessie.silver.*')
print('Next step: Run 03_gold_layer.ipynb')